# 06 - Power BI Ready Views

## Purpose

Validate reporting shapes that Power BI should consume. Persistent SQL views should be created from the SQL analytics endpoint using scripts in the sql folder.

In [ ]:
from pyspark.sql import functions as F

fact = spark.table("fact_transaction")
dim_customer = spark.table("dim_customer")
dim_account = spark.table("dim_account")
dim_product = spark.table("dim_product")
dim_branch = spark.table("dim_branch")
dim_date = spark.table("dim_date")

In [ ]:
transaction_detail = (
    fact.alias("f")
    .join(dim_date.alias("d"), "date_key")
    .join(dim_customer.alias("c"), "customer_key")
    .join(dim_account.alias("a"), "account_key")
    .join(dim_product.alias("p"), "product_key")
    .join(dim_branch.alias("b"), "branch_key")
    .select("f.transaction_id", "d.date_value", "d.year_month", "c.customer_segment", "c.customer_status", "a.account_type", "a.account_status", "p.product_name", "p.product_category", "b.branch_name", "b.region", "f.transaction_channel", "f.transaction_type", "f.transaction_status", "f.transaction_amount", "f.absolute_transaction_amount", "f.is_posted")
)
transaction_detail.createOrReplaceTempView("vw_powerbi_transaction_detail")
display(transaction_detail.limit(20))

In [ ]:
monthly_summary = (
    transaction_detail
    .filter(F.col("is_posted") == True)
    .groupBy("year_month", "product_category", "region")
    .agg(F.countDistinct("transaction_id").alias("transaction_count"), F.sum("absolute_transaction_amount").alias("total_transaction_amount"), F.avg("absolute_transaction_amount").alias("average_transaction_amount"))
    .orderBy("year_month", "product_category", "region")
)
monthly_summary.createOrReplaceTempView("vw_powerbi_monthly_summary")
display(monthly_summary)

In [ ]:
customer_summary = (
    dim_customer.alias("c")
    .join(dim_account.alias("a"), "customer_id", "left")
    .groupBy("c.customer_id", "c.customer_full_name", "c.customer_segment", "c.customer_status", "c.city", "c.state")
    .agg(F.countDistinct("a.account_id").alias("account_count"), F.sum("a.current_balance").alias("total_current_balance"))
    .orderBy("c.customer_segment", "c.customer_full_name")
)
customer_summary.createOrReplaceTempView("vw_powerbi_customer_summary")
display(customer_summary)